# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('/Users/abhishekkarthikakunuru/Documents/GitHub/dataviz-exercises-abhishek/data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [3]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [4]:
# Task — Multi-series line with highlight
HIGHLIGHT = 'China'
GREY = '#EEEEEE'
COLOR = '#C0392B'

asia = df[df['Region'] == 'Asia']

fig = go.Figure()

# 1) Context: every other Asian country in thin light grey
for country, g in asia.groupby('Country'):
    if country == HIGHLIGHT:
        continue
    fig.add_trace(go.Scatter(
        x=g['Year'], y=g['CO2_Mt'], mode='lines',
        line=dict(color=GREY, width=1.5),
        name=country, showlegend=False,
        hovertemplate=f'{country}<br>%{{x}}: %{{y:,.0f}} Mt<extra></extra>'
    ))

# 2) Highlight: drawn last so it sits on top
h = asia[asia['Country'] == HIGHLIGHT].sort_values('Year')
fig.add_trace(go.Scatter(
    x=h['Year'], y=h['CO2_Mt'], mode='lines',
    line=dict(color=COLOR, width=4),
    name=HIGHLIGHT, showlegend=False,
    hovertemplate=f'{HIGHLIGHT}<br>%{{x}}: %{{y:,.0f}} Mt<extra></extra>'
))

# 3) Direct label at the end of the highlighted line (no legend)
first, last = h.iloc[0], h.iloc[-1]
fig.add_annotation(
    x=last['Year'], y=last['CO2_Mt'],
    text=f"<b>{HIGHLIGHT}</b><br>{last['CO2_Mt']:,.0f} Mt",
    showarrow=False, xanchor='left', xshift=8,
    font=dict(color=COLOR, size=14)
)
fig.add_annotation(
    x=first['Year'], y=first['CO2_Mt'],
    text=f"{first['CO2_Mt']:,.0f} Mt", showarrow=False,
    xanchor='left', yanchor='top', yshift=-8,
    font=dict(color=COLOR, size=12)
)

# 4) Insight title + clean styling
mult = last['CO2_Mt'] / first['CO2_Mt']
fig.update_layout(
    title=dict(
        text=f"<b>China's CO2 emissions grew {mult:.1f}x since 2000</b>, "
             f"while the other Asian economies stayed far below",
        x=0, xanchor='left', font=dict(size=20)
    ),
    template='simple_white',
    plot_bgcolor='white',
    showlegend=False,
    margin=dict(l=60, r=110, t=90, b=50),
    height=500,
    xaxis=dict(showgrid=False, showline=False, tickangle=0, title=None,
               dtick=2, range=[2000, 2022.2]),
    yaxis=dict(showgrid=False, showline=False, title='CO2 emissions (Mt)',
               tickformat=',', rangemode='tozero'),
)
fig.show()
